In [1]:
import sys
import numpy as np
from game_helper import GameHelper  
from replay_buffer import ReplayBuffer
from configs import Config
import random
import tensorflow as tf
from game.game import Game
from model_helper import ModelHelper
from dqn_trainer import DqnTrainer
from dqn_trainer import create_model, compile_model

2025-10-26 17:52:52.647329: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
version = tf.__version__
print("TensorFlow version:", version)
python_version = sys.version
print("Python version:", python_version)

TensorFlow version: 2.20.0
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]


In [3]:
game = Game(verbose=False)
replay = ReplayBuffer()

In [4]:
model=create_model()
compile_model(model)

/opt/conda/lib/python3.11/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-10-26 17:52:55.672830: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,310 (71.52 KB)

 Trainable params: 18,310 (71.52 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
trainer = DqnTrainer(model=model)
trainer.train_model(game=game, replay=replay, debug=True)

Game 1 started
raw state: [ 4.  4.  4.  4.  4.  4.  0.  4.  4.  4.  4.  4.  4.  0.  0.  0. 24. 24.
  1.  1.]
normalized: [0.08333334 0.08333334 0.08333334 0.08333334 0.08333334 0.08333334
 0.         0.08333334 0.08333334 0.08333334 0.08333334 0.08333334
 0.08333334 0.         0.         0.         0.5        0.5
 0.16666667 0.16666667]
q_values: [ 0.04237876  0.02373407 -0.00422982 -0.08061638  0.04143279  0.12468144]
raw state: [ 4.  4.  4.  4.  4.  0.  1.  5.  5.  5.  4.  4.  4.  0.  0.  1. 21. 27.
  1.  1.]
normalized: [0.08333334 0.08333334 0.08333334 0.08333334 0.08333334 0.
 0.02083333 0.10416666 0.10416666 0.10416666 0.08333334 0.08333334
 0.08333334 0.         0.         0.02083333 0.4375     0.5625
 0.16666667 0.16666667]
q_values: [ 0.04244831  0.01778595  0.00482044 -0.09692977  0.04869989  0.13254431]
raw state: [ 4.  4.  4.  4.  4.  0.  1.  0.  6.  6.  5.  5.  5.  0.  0. -4. 21. 27.
  1.  2.]
normalized: [ 0.08333334  0.08333334  0.08333334  0.08333334  0.08333334  0.
  0

In [6]:
from model_debug import debug_single_batch

test_states = []
test_legal_masks = []
game.initialize()

for _ in range(5):
    features = ModelHelper.build_features(game.get_board(), game.current_player, game.game_finish)
    normalized_state = ModelHelper.normalize_single_fixed(features)
    legal_actions = game.get_playable_pits()
    
    mask = np.zeros(6)
    for action in legal_actions:
        if 0 <= action <= 5: 
            mask[action] = 1
        elif 7 <= action <= 12:
            mask[action - 7] = 1
            
    test_states.append(normalized_state)
    test_legal_masks.append(mask)
    
    action = random.choice(legal_actions)
    game.play(action)
    
test_states = np.array(test_states)
test_legal_masks = np.array(test_legal_masks)

print("Debug output for a batch of 5 game states:")
debug_single_batch(model, test_states, test_legal_masks)

Debug output for a batch of 5 game states:
DEBUG BATCH: loss=None, illegal_before=0.000, illegal_after=0.000, entropy_mean=1.6122
Sample 0:
  logits: [3.31  3.374 3.756 3.629 3.831 3.835]
  masked: [3.31  3.374 3.756 3.629 3.831 3.835]
  probs: [0.119 0.127 0.187 0.164 0.201 0.202], entropy: 1.7713
Sample 1:
  logits: [3.305 3.364 3.746 3.635 3.833 3.835]
  masked: [ 3.305e+00  3.364e+00 -1.000e+09  3.635e+00  3.833e+00  3.835e+00]
  probs: [0.146 0.155 0.    0.203 0.248 0.248], entropy: 1.5852
Sample 2:
  logits: [3.481 3.532 3.934 3.808 4.044 4.062]
  masked: [3.481 3.532 3.934 3.808 4.044 4.062]
  probs: [0.117 0.123 0.184 0.162 0.205 0.209], entropy: 1.7671
Sample 3:
  logits: [3.504 3.545 3.919 3.828 4.046 4.066]
  masked: [ 3.504e+00  3.545e+00 -1.000e+09 -1.000e+09  4.046e+00  4.066e+00]
  probs: [0.181 0.189 0.    0.    0.312 0.318], entropy: 1.3522
Sample 4:
  logits: [3.354 3.438 3.796 3.725 3.908 3.95 ]
  masked: [ 3.354e+00  3.438e+00  3.796e+00  3.725e+00 -1.000e+09  3.950